# Extended Memristor Code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import subprocess
import ltspice
from scipy.optimize import minimize
from scipy.interpolate import interp1d
from pyDOE import lhs
import os
from skopt import gp_minimize
from skopt.space import Real
from dtaidistance import dtw
from scipy.stats import zscore

## Plot Experimental Data

In [ ]:
def process_file(file_path):
    """
    Load data from a CSV file and compute voltage, current, time,
    and their respective integrals over time.

    Parameters:
        file_path (str): Path to the CSV file.

    Returns:
        v (ndarray): Voltage values
        i (ndarray): Current values
        t (ndarray): Time values
        f (ndarray): Integral of voltage over time
        q (ndarray): Integral of current over time
    """
    data = np.loadtxt(file_path, delimiter=',')

    # Extract columns (adjust indices if needed)
    v = data[:, 0]  # First column: Voltage
    i = data[:, 2]  # Third column: Current
    t = data[:, 4]  # Fifth column: Time

    # Compute integrals using cumulative sum and time gradient
    f = np.cumsum(v * np.gradient(t))  # Voltage integral w.r.t. time
    q = np.cumsum(i * np.gradient(t))  # Current integral w.r.t. time

    return v, i, t, f, q


def plot_graphs(v, i, t, f, q):
    """
    Plot voltage, current, and their integrals against time.

    Parameters:
        v (ndarray): Voltage values
        i (ndarray): Current values
        t (ndarray): Time values
        f (ndarray): Voltage integral over time
        q (ndarray): Current integral over time
    """
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))

    # Voltage vs Time
    axs[0, 0].scatter(t, v, label="v(t)")
    axs[0, 0].set_title("Voltage vs Time")
    axs[0, 0].set_xlabel("Time")
    axs[0, 0].set_ylabel("Voltage")
    axs[0, 0].grid(True)

    # Current vs Voltage
    axs[0, 1].plot(v, i, label="i(v)")
    axs[0, 1].set_title("Current vs Voltage")
    axs[0, 1].set_xlabel("Voltage")
    axs[0, 1].set_ylabel("Current")
    axs[0, 1].grid(True)

    # Current vs Time
    axs[0, 2].scatter(t, i, label="i(t)")
    axs[0, 2].set_title("Current vs Time")
    axs[0, 2].set_xlabel("Time")
    axs[0, 2].set_ylabel("Current")
    axs[0, 2].grid(True)

    # Voltage Integral vs Time
    axs[1, 0].scatter(t, f, label="f(t)")
    axs[1, 0].set_title("Voltage Integral vs Time")
    axs[1, 0].set_xlabel("Time")
    axs[1, 0].set_ylabel("Voltage Integral")
    axs[1, 0].grid(True)

    # Current Integral vs Time
    axs[1, 1].scatter(t, q, label="q(t)")
    axs[1, 1].set_title("Current Integral vs Time")
    axs[1, 1].set_xlabel("Time")
    axs[1, 1].set_ylabel("Current Integral")
    axs[1, 1].grid(True)

    # Remove unused subplot
    fig.delaxes(axs[1, 2])

    plt.tight_layout()
    plt.show()

    

def plot_voltage_current(time, voltage, current):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

    # Voltage vs Time
    ax1.plot(time, voltage, label='Voltage', color='b')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Voltage (V)')
    ax1.set_title('Voltage as a Function of Time')
    ax1.legend()
    ax1.grid(True)

    # Current vs Voltage
    ax2.plot(voltage, current, label='Current vs Voltage', color='r')
    ax2.set_xlabel('Voltage (V)')
    ax2.set_ylabel('Current (A)')
    ax2.set_title('Current as a Function of Voltage')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
# Insert data you want to fit and analyze

file_paths = [
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_4V_0.1V_s_sin_1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_4V_0.1V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_10V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_5V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_1V_s_lin__1.txt",
    r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_0.5V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_0.05V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_0.2V_s_lin__1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_0.1V_s_sin_1.txt",
    #r"D:\Davide_Rossetti\Python Extended Memristor\Simulations_20_03\IV_for_model_set5\IV_3layer_CB32x32_devT11B15_3V_0.1V_s_lin__1.txt"
]

for file_path in file_paths:
    voltage, current, time, f, q = process_file(file_path)
    plot_graphs(voltage, current, time, f, q)

Consideriamo una media tra i diversi cicli di misurazioni

In [ ]:
# Number of cycles to average
n_cycles = 5

# Total number of data points
n_total_points = len(voltage)

# Points per cycle (assuming equal length cycles)
n_points_per_cycle = n_total_points // n_cycles

# Reshape arrays into (n_cycles x n_points_per_cycle)
voltage_matrix = voltage[:n_cycles * n_points_per_cycle].reshape(n_cycles, n_points_per_cycle)
current_matrix = current[:n_cycles * n_points_per_cycle].reshape(n_cycles, n_points_per_cycle)
time_matrix = time[:n_cycles * n_points_per_cycle].reshape(n_cycles, n_points_per_cycle)

# Compute the average cycle (point-by-point mean)
voltage_avg = np.mean(voltage_matrix, axis=0)
current_avg = np.mean(current_matrix, axis=0)
time_avg = np.mean(time_matrix, axis=0)

# Estimate frequency as the reciprocal of the maximum time value
frequency = 1 / np.max(time_avg)

# Plot average I–V curve
plt.plot(voltage_avg, current_avg, label="Average Cycle", color='black')
plt.xlabel("Voltage (V)")
plt.ylabel("Current (A)")
plt.title(f"Mean Sweep over {n_cycles} Cycles")
plt.grid(True)
plt.legend()
plt.show()

## Automation Spice

In [ ]:
def generate_lhs_samples(num_params, num_samples, param_ranges):
    
    # num_params: Number of parameters to sample.
    # num_samples: Number of samples to generate.
    # param_ranges: List of tuples specifying the range of each parameter [(min, max), ...].
    # Returns: A sample matrix (num_samples x num_params).

    # Generate normalized samples between 0 and 1
    samples = lhs(num_params, num_samples)
    
    # Scale samples to parameter ranges
    scaled_samples = np.zeros_like(samples)
    for i in range(num_params):
        min_val, max_val = param_ranges[i]
        scaled_samples[:, i] = samples[:, i] * (max_val - min_val) + min_val

    return scaled_samples



###############################################################################################
def update_netlist(params, netlist_file, frequency):

    try:
        with open(netlist_file, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        print(f"The file '{netlist_file}' was not found.")
        return
    period = 1 / frequency
    t1 = period / 4
    t2 = period / 2
    t3 = period * 3 / 4
    
    # Modify parameters in the netlist, adapt to the specific netlist everytime
    # lines[1] = f"R1 N002 N001 {params[0]}\n"
    
    #lines[1] = f"V1 N1 0 PWL(0 0 {t1} 4 {t2} 0 {t3} -4 {period} 0)\n"
    lines[1] = f"V1 N1 0 PWL(0 0 {t1} 3 {t2} 0 {t3} -3 {period} 0)\n"
    lines[8] = f".tran 0 {period} 0 1m\n"
    lines[9] = f".param k={params[6]} c={params[2]} a={params[0]} f={frequency} phi={params[12]} h={params[5]} b={params[1]} g={params[4]} p={params[7]} d={params[3]} y={params[9]} x={params[8]} I0={params[10]} eta={params[11]} I1={params[13]} eta1={params[14]}\n"
    

    with open(netlist_file, 'w') as file:
        file.writelines(lines)
        
        
        
####################################################################################################
def run_spice_simulation(netlist_file):
    ltspice_path = r"C:\Users\Davide\AppData\Local\Programs\ADI\LTspice\LTspice.exe"   #insert the path for spice
    
    # Run LTspice in batch mode
    process = subprocess.run([ltspice_path, "-b", netlist_file], capture_output=True, text=True)
    
    
    
####################################################################################################
def read_simulation_results(raw_file):
    
    l = ltspice.Ltspice(raw_file)
    l.parse()  # Parse the raw file
    
    time = l.get_time()
    voltage = l.get_data('V(n1)')
    current = l.get_data('I(V1)')  # Replace with the correct node name
    
    return time, voltage, current


####################################################################################################
def compute_mse(simulated_v, experimental_v, simulated_i, experimental_i, t):
    
    error_v = np.mean((simulated_v - experimental_v)**2)
    error_i = np.sqrt(np.mean(abs(simulated_i - experimental_i)**2))
    relative_error = abs(np.mean((simulated_i - experimental_i) / experimental_i))
    
    difference = np.abs(simulated_i - experimental_i)
    abc = np.trapz(difference, x=t)
    
    return error_i, relative_error, error_v, abc


#####################################################################################################
def minimum_error(errors, lhs_samples):
    
    best_index = np.argmin(errors)  # Get the index of the sample with the smallest error
    best_initial_params = lhs_samples[best_index]  # Take the best initial parameters

    print(f"The best initial parameters are: {best_initial_params}")
    print(f"The corresponding error is: {errors[best_index]}")
    
    return best_index, best_initial_params


## Bayesian Method for Parameters Optimization

In [ ]:
# Objective Function

def objective_function(params, scale_factors):
        rescaled_params = params * scale_factors
        update_netlist(rescaled_params, netlist_file, frequency)
        run_spice_simulation(netlist_file)
        time_lo, voltage_lo, current_lo = read_simulation_results(raw_file)
        current_lo = - current_lo
        
        common_time = np.linspace(0.021, min(time[-1], time_lo[-1]), 1000)
        v_exp = interp1d(time, voltage)(common_time)
        i_exp = interp1d(time, current)(common_time)
        voltage_lo = interp1d(time_lo, voltage_lo)(common_time)
        current_lo = interp1d(time_lo, current_lo)(common_time)
        current_lo[current_lo > 5e-6] = 5e-6
        
        t_exp = common_time
        global global_iter
        global_iter += 1
        print(f"[Iteration {global_iter}]")
        return np.sqrt(np.mean((current_lo - i_exp) ** 2))
    
    
# Function for parameter scaling

def create_scale_factors(vector):
    # Compute the next power of 10 for each element of the vector
    scale_factors = np.power(10, np.ceil(np.log10(np.abs(vector))))
    return scale_factors


#########################################################################################################

# Method 1: Bayesian Optimization

def Bayesian_Optimization(bounds_normalized, scale_factors, best_initial_params_normalized=None):
    
    # Define the normalized parameter space for gp_minimize
    space = [Real(b[0], b[1]) for b in bounds_normalized]

    # If a set of initial parameters from `lhs` or another method is provided, use it as the starting point
    x0 = [best_initial_params_normalized.tolist()] if best_initial_params_normalized is not None else None

    # Run Bayesian optimization
    result = gp_minimize(lambda params: objective_function(params, scale_factors),
                         space, 
                         x0=x0,                # Optional initial point (e.g., from `lhs`)
                         n_calls=15,           # Number of function evaluations
                         n_initial_points=3,
                         random_state=0)       # For reproducibility

    # Rescale optimized parameters back to the original scale
    optimized_params = np.array(result.x) * scale_factors

    # Print results
    print("Results with Bayesian Optimization:")
    print(f"Optimized parameters: {optimized_params}")
    print(f"Minimum objective function value: {result.fun}")
    
    return optimized_params, result.fun


## Gradient Method

In [ ]:
# Method 2: Gradient Optimization

def Gradient_Method(best_initial_params_normalized, bounds_normalized, scale_factors, method):
    
    # Perform optimization using L-BFGS-B
    result = minimize(objective_function, 
                      best_initial_params_normalized, 
                      args=(scale_factors,), 
                      method=method, 
                      bounds=bounds_normalized,
                      tol=1e-8,
                      options={'maxiter': 10, 'maxfun': 20, 'disp': True})

    # Rescale optimized parameters back to the original scale
    optimized_params = result.x * scale_factors

    # Print results
    print(f"Optimized parameters (result.x): {optimized_params}")
    print(f"Minimum objective function value (result.fun): {result.fun}")
    print(f"Was the optimization successful? (result.success): {result.success}")
    print(f"Termination message (result.message): {result.message}")
    print(f"Number of iterations (result.nit): {result.nit}")
    if 'jac' in result:
        print(f"Objective function gradient at the optimal point (result.jac): {result.jac}")
        
    return optimized_params, result.fun


## Execution Code

In [ ]:
# Execution Tab
num_params = 15 
num_samples = 4

# Order parameters
# a={p[0]} b={p[1]} c={p[2]} d={p[3]} g={p[4]} h={p[5]} k={p[6]} p={p[7]} x={p[8]} y={p[9]} I0={p[10]} eta={p[11]} phi={p[12]} I1={p[13]} eta1={p[14]}
param_ranges = [(3e-12, 6e-12), (6, 8), (4e-7, 6e-7), (1e-1, 2e-1), (14, 16), (1e-7, 2e-7), 
                (0.4, 1), (7, 9), (2, 4), (1e-2, 3e-2), (3e-12, 10e-12), (2, 4), (1, 1.5),
               (5e-12,15e-12), (-4,-2)]

num_original = len(time)
error_vector = np.zeros((num_samples))
best_initial_params = np.zeros((num_params))

# Generate samples to explore the parameter space
samples_lhs = generate_lhs_samples(num_params, num_samples, param_ranges)
netlist_file = r"C:\Users\Davide\Desktop\Extended Memristor Github\Modello2k25_modifiable.net"
raw_file = r"C:\Users\Davide\Desktop\Extended Memristor Github\Modello2k25_modifiable.raw"

for i in range(num_samples):
    print(i+1)
    update_netlist(samples_lhs[i, :], netlist_file, frequency)
    run_spice_simulation(netlist_file)
    time_spice, voltage_spice, current_spice = read_simulation_results(raw_file)
    
    common_time = np.linspace(0.021, min(time[-1], time_spice[-1]), 1000)
    v_exp = interp1d(time, voltage)(common_time)
    i_exp = interp1d(time, current)(common_time)
    voltage_spice = interp1d(time_spice, voltage_spice)(common_time)
    current_spice = interp1d(time_spice, current_spice)(common_time)
    t_exp = common_time
        
    plot_voltage_current(common_time, voltage_spice, -current_spice)
    error_mse, relative_error, error_voltage, abc_distance = compute_mse(voltage_spice, v_exp, -current_spice, i_exp, t_exp)
    error_vector[i] = error_mse
    print('RMSE error:', error_mse)
    #print("Area Between Curves:", abc_distance)
    print("Relative error:", relative_error)
    print("Voltage error:", error_voltage)

    
best_index, best_initial_params = minimum_error(error_vector, samples_lhs)
param_names = ["a", "b", "c", "d", "g", "h", "k", "p", "x", "y", "I0", "eta", "phi", "I1", "eta1"]
print("Best initial parameters:")
for name, value in zip(param_names, best_initial_params):
    print(f"  {name} = {value:.6g}")
print("Best sample index:", best_index)
print("End")


In [ ]:
scale_factors = create_scale_factors(best_initial_params)
bounds_normalized = [(low / sf, high / sf) for (low, high), sf in zip(param_ranges, scale_factors)]
best_initial_params_normalized = best_initial_params / scale_factors

In [ ]:
global_iter = 0
optimized_params_gradient, fun_gradient = Gradient_Method(best_initial_params_normalized, bounds_normalized, scale_factors, "L-BFGS-B")

In [ ]:
global_iter = 0
optimized_params_bayesian, fun_bayesian = Bayesian_Optimization(bounds_normalized, scale_factors, best_initial_params_normalized)

In [ ]:
global_iter = 0
optimized_params_bayesian_normalized = optimized_params_bayesian / scale_factors
optimized_params_hybrid, fun_hybrid = Gradient_Method(optimized_params_bayesian_normalized, bounds_normalized, scale_factors,"L-BFGS-B")

In [ ]:
update_netlist(optimized_params_gradient, netlist_file, frequency)
run_spice_simulation(netlist_file)
time_spice, voltage_spice, current_spice = read_simulation_results(raw_file)
current_spice = -current_spice
current_spice[current_spice>5e-6] = 5e-6
plot_voltage_current(time_spice, voltage_spice, current_spice)
plot_voltage_current(time, voltage, current)